# 06 · ΔPSI per exon

Notebook 05 tested every exon. This notebook pulls out the two PICALM exons: junction reads, PSI per replicate, ΔPSI and FDR, together with the QKI knockdown level from notebook 04. These numbers make panel d of the figure.

| Exon | GENCODE ID | hg19 (rMATS) | GRCh38 |
| --- | --- | --- | --- |
| 150-nt exon | ENSE00000989276.1 | chr11:85,701,292-85,701,442 | chr11:85,990,249-85,990,399 |
| 24-nt exon | ENSE00001416869.1 | chr11:85,689,112-85,689,136 | chr11:85,978,069-85,978,093 |

Both are on the minus strand; starts are 0-based, as in rMATS.

On Google Colab, run the next cell first in every session. It copies this repository onto the Colab machine and, with `SAVE_TO_DRIVE = True`, saves results and the figure to `My Drive/CS3_workflow` (Colab asks for permission to connect your Drive). `LITE = True` uses a small subset of the data that fits free Colab. On your own computer the cell does nothing.

In [ ]:
import sys

if "google.colab" in sys.modules:
    import os
    import subprocess
    REPO = "https://github.com/yeolab/encode-rbp-case-study"
    CLONE = "/content/encode-rbp-case-study"
    LITE = True    # True: small subset of the data, fits free Colab (about 1 h). False: all the data (5-8 h, ~50 GB)
    SAVE_TO_DRIVE = True    # results and the figure in My Drive/CS3_workflow; False: on this Colab machine only
    if not os.path.isdir(CLONE):
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPO, CLONE], check=True)
    WORKFLOW = CLONE + "/notebooks/workflow"
    OUTPUT = None
    if SAVE_TO_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        OUTPUT = "/content/drive/MyDrive/CS3_workflow"
    sys.path.insert(0, WORKFLOW)
    import colab_setup
    colab_setup.setup(WORKFLOW, lite=LITE, output_folder=OUTPUT)

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

sys.path.insert(0, ".")
import workflow_utils as U

ROOT, DATA = U.folders()
RESULTS = ROOT / "results" / "workflow"
API_CACHE = DATA / "downloads" / "rbpark_api"
print("case study folder:", ROOT)
print("data folder:", DATA)
RMATS_DIR = U.data_dirs(DATA)["rmats"]       # in lite mode: the full run's tables in lite_data/

In [ ]:
level = pd.read_csv(RESULTS / "04_knockdown_level.tsv", sep="\t").set_index("knockdown")
knockdown, rows = {}, []
for key in U.KNOCKDOWNS:
    knockdown[key] = {"log2fc": float(level.loc[key, "QKI log2FC"])}
    for exon, info in U.EXONS.items():
        row = U.exon_row(RMATS_DIR / key / "SE.MATS.JC.txt", exon)
        st = U.exon_stats(row)
        knockdown[key][exon] = st
        rows.append({"knockdown": key, "exon": info["name"], "PSI control": st["psi_ctl"], "PSI knockdown": st["psi_kd"],
                     "dPSI": st["dpsi"], "FDR": st["fdr"], "inclusion reads (ctl | KD)": (st["IJC_ctl"], st["IJC_kd"]),
                     "skipping reads (ctl | KD)": (st["SJC_ctl"], st["SJC_kd"]), "note": row["note"]})
(RESULTS / "06_picalm_knockdown.json").write_text(json.dumps(knockdown, indent=1))
pd.DataFrame(rows)

In [ ]:
fig = U.plot_psi(knockdown)

QKI knockdown raises the 150-nt exon's PSI (+0.22 in K562, +0.47 in HepG2) and lowers the 24-nt exon's (−0.10 and −0.55). Both changes are larger in HepG2, where less QKI is left. FDR is below 1e-10 in every case.